# Main Agent Test Notebook

This notebook tests the MainAgent with error detection.

Tests both successful queries and error scenarios with tool failures.

In [1]:
import os
import sys
from pathlib import Path
import logging

# Configure Redis for local development (before other imports)
os.environ['REDIS_URL'] = 'redis://localhost:6379'

# Disable verbose logging
logging.basicConfig(level=logging.WARNING)
logging.getLogger('httpx').setLevel(logging.WARNING)
logging.getLogger('app').setLevel(logging.WARNING)

# Add backend to path
backend_dir = Path.cwd()
if backend_dir.name != 'backend':
    backend_dir = backend_dir / 'backend'
sys.path.insert(0, str(backend_dir))

from dotenv import load_dotenv
load_dotenv()

print("✅ Environment loaded")

✅ Environment loaded


In [2]:
from langchain_openai import ChatOpenAI
from app.agents.main_agent import MainAgent
from langgraph.checkpoint.memory import MemorySaver

# Initialize LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Database path
db_path = backend_dir / "app" / "resource" / "art.db"

print(f"Database: {db_path}")
print(f"Exists: {db_path.exists()}")

e:\Final Year Project\data_exploration_agent\backend\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Database: e:\Final Year Project\data_exploration_agent\backend\app\resource\art.db
Exists: True


In [3]:
# Initialize MainAgent
print("Initializing MainAgent...")

agent = MainAgent(
    llm=llm,
    db_path=str(db_path),
    use_postgres_checkpointer=False  # Use MemorySaver for testing
)

print("✅ MainAgent initialized!")
print(f"📊 Graph saved to: {agent.logs_dir}/main_agent_graph.png")

Initializing MainAgent...
✅ MainAgent initialized!
📊 Graph saved to: e:\Final Year Project\data_exploration_agent\backend\logs/main_agent_graph.png


## Test Queries

### 1. Success Case
Test a normal successful query

In [4]:
# Setup checkpointer and config
config = {
    "configurable": {"thread_id": "test-success"},
    "recursion_limit": 50
}

print("✅ Testing successful query...")

✅ Testing successful query...


In [5]:
question = "Which genre has the oldest painting?"
print(f"\n{'='*60}")
print(f"Question: {question}")
print('='*60)

# Stream the execution
for step in agent.graph.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="updates",
    config=config
):
    node_name = list(step.keys())[0] if step else "unknown"
    print(f"\n[{node_name}]")
    
    # Show key state info
    if node_name in step:
        node_data = step[node_name]
        if "current_step_index" in node_data:
            print(f"  Step index: {node_data['current_step_index']}")
        if "feedback" in node_data and node_data["feedback"]:
            print(f"  ⚠️  Feedback: {node_data['feedback']}")
        if "error_details" in node_data and node_data['error_details']:
            print(f"  ❌ Errors: {len(node_data['error_details'])}")

print("\n" + "="*60)
print("✅ Query completed!")
print("="*60)


Question: Which genre has the oldest painting?

[planner]
  Step index: 0

[process_query]
  Step index: 1

[tools]

[explainer]
  Step index: 1

[process_query]

[finalizer]

✅ Query completed!


### 2. Error Case - Invalid Table
Test error detection when querying a non-existent table

In [6]:
# New config for error test
error_config = {
    "configurable": {"thread_id": "test-error"},
    "recursion_limit": 50
}

error_question = "Query the xyz_fake_table"
print(f"\n{'='*60}")
print(f"Error Test: {error_question}")
print('='*60)

# Stream execution and watch for errors
for step in agent.graph.stream(
    {"messages": [{"role": "user", "content": error_question}]},
    stream_mode="updates",
    config=error_config
):
    node_name = list(step.keys())[0] if step else "unknown"
    print(f"\n[{node_name}]")
    
    if node_name in step:
        node_data = step[node_name]
        
        # Highlight error information
        if "feedback" in node_data and node_data["feedback"]:
            print(f"  ⚠️  FEEDBACK: {node_data['feedback']}")
            
        if "error_details" in node_data and node_data['error_details']:
            print(f"  ❌ ERROR DETAILS:")
            for err in node_data['error_details']:
                print(f"     Tool: {err.get('tool_name')}")
                print(f"     Type: {err.get('error_type')}")
                print(f"     Message: {err.get('error_message')}")
                print(f"     Recoverable: {err.get('recoverable')}")
                print(f"     Detection: {err.get('detection_method')}")
        
        if "current_step_index" in node_data:
            print(f"  📍 Step index: {node_data['current_step_index']}")

print("\n" + "="*60)
print("✅ Error test completed!")
print("="*60)


Error Test: Query the xyz_fake_table

[planner]
  📍 Step index: 0


2026-01-06 01:18:39,625 - app.agents.main_agent - WARNING - ✅ JSON Error detected in tool data_exploration_tool: Failed to generate SQL: Table validation failed: The table mentioned does not exist.
ERROR:app.agents.main_agent:Tool execution failed. Rolling back step index from 1 to 0
2026-01-06 01:18:39,625 - app.agents.main_agent - ERROR - Tool execution failed. Rolling back step index from 1 to 0
ERROR:app.agents.main_agent:Feedback: Tool execution error in step 1: data_exploration_tool
2026-01-06 01:18:39,631 - app.agents.main_agent - ERROR - Feedback: Tool execution error in step 1: data_exploration_tool



[process_query]
  📍 Step index: 1

[tools]
  ⚠️  FEEDBACK: Tool execution error in step 1: data_exploration_tool
  ❌ ERROR DETAILS:
     Tool: data_exploration_tool
     Type: validation_error
     Message: Failed to generate SQL: Table validation failed: The table mentioned does not exist.
     Recoverable: False
     Detection: json
  📍 Step index: 0

[explainer]
  ⚠️  FEEDBACK: Tool execution error in step 1: data_exploration_tool
  ❌ ERROR DETAILS:
     Tool: data_exploration_tool
     Type: validation_error
     Message: Failed to generate SQL: Table validation failed: The table mentioned does not exist.
     Recoverable: False
     Detection: json
  📍 Step index: 0

[process_query]
  📍 Step index: 1

[__interrupt__]

✅ Error test completed!


### 3. Check Final States
Compare the final states of both runs

In [ ]:
# Success case state
success_state = agent.graph.get_state(config)
print("Success Case Final State:")
print(f"  Messages: {len(success_state.values.get('messages', []))}")
print(f"  Steps: {len(success_state.values.get('steps', []))}")
print(f"  Current step index: {success_state.values.get('current_step_index', 0)}")
print(f"  Feedback: {success_state.values.get('feedback')}")
print(f"  Error details: {len(success_state.values.get('error_details', []))}")

print("\n" + "-"*60 + "\n")

# Error case state
error_state = agent.graph.get_state(error_config)
print("Error Case Final State:")
print(f"  Messages: {len(error_state.values.get('messages', []))}")
print(f"  Steps: {len(error_state.values.get('steps', []))}")
print(f"  Current step index: {error_state.values.get('current_step_index', 0)}")
print(f"  Feedback: {error_state.values.get('feedback')}")
print(f"  Error details count: {len(error_state.values.get('error_details', []))}")

# Show detailed error info
error_details = error_state.values.get('error_details', [])
if error_details:
    print("\n  Detailed Errors:")
    for i, err in enumerate(error_details, 1):
        print(f"    Error {i}:")
        print(f"      Tool: {err.get('tool_name')}")
        print(f"      Type: {err.get('error_type')}")
        print(f"      Message: {err.get('error_message')}")
        print(f"      Recoverable: {err.get('recoverable')}")
        print(f"      Detection Method: {err.get('detection_method')}")

### 4. Verify Step Rollback
Ensure that step index doesn't increment on error

In [ ]:
print("Verification Results:")
print("=" * 60)

# Check if error was detected
has_feedback = error_state.values.get('feedback') is not None
has_errors = len(error_state.values.get('error_details', [])) > 0

print(f"✓ Error detected via feedback: {has_feedback}")
print(f"✓ Error details captured: {has_errors}")

# Check error type
if has_errors:
    err = error_state.values.get('error_details', [])[0]
    is_json_detected = err.get('detection_method') == 'json'
    print(f"✓ JSON error detection used: {is_json_detected}")
    print(f"✓ Error type classified: {err.get('error_type')}")
    print(f"✓ Recoverable flag set: {err.get('recoverable') is not None}")

print("\n" + "="*60)
print("🎉 Test suite complete!")